# Experimental Validation Analysis

This notebook reproduces the experimental-validation panel used in the manuscript. It compares normalized relative enrichment values for sequences sampled from different DCA model variants, stratified by sequence-divergence batch.

The notebook is organized as a reproducibility artifact: input vectors are stored explicitly in the notebook, the Gaussian mixture fits are generated from those vectors, and the final publication figure is written to `images/experiment.pdf`.


## Imports and Plot Configuration

Load numerical, plotting, and Gaussian mixture model dependencies. The shared color palette is defined here so all panels use the same model-to-color mapping.


In [ ]:
# ============================================================================
# IMPORTS AND PLOT CONFIGURATION
# ============================================================================
# Notebook purpose:
# - Visualize experimental validation data for multiple DCA model variants.
# - Plot normalized relative enrichment distributions across sequence-divergence
#   bins and fit 2-component Gaussian Mixture Models (GMM) to highlight modes.
# - Annotate fraction of functional sequences per sample and save
#   publication-ready figures under the 'images/' folder.
#
# Conventions:
# - Variable names: norm_re_*_batch# store normalized relative enrichment (float).
# - functional_*_batch# are binary arrays (1 = functional, 0 = non-functional).
# - Batches 1..3 correspond to increasing sequence-divergence intervals.
# ============================================================================

import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture

# Define color palette for the four DCA model variants
palette = sns.color_palette("deep", 4)
colors = {
    'highS': palette[0],    # High entropy model - Cool blue
    'bmDCA': palette[1],    # Boltzmann machine DCA - Purple-pink
    'dense': palette[2],    # Dense DCA - Pink-orange
    'eaDCA': palette[3]     # Extreme attributes DCA - Warm red
}

# Configure LaTeX-style plotting aesthetics for publication quality
plt.rcParams.update({
    "text.usetex": True,
})

print("Setup complete!")

output_path = "images/"
os.makedirs(output_path, exist_ok=True)

## Experimental Measurements

Define the normalized relative enrichment arrays and binary functional labels extracted from the experimental CSV files. The batch suffixes correspond to increasing sequence-divergence intervals used in the paper.


In [ ]:
# Extracted from uploaded CSV files:
# - `norm_re_*`: column 'M9c Norm R.E.' (normalized relative enrichment).
# - `functional_*`: binary functional classification derived from
#   'Null Mode 3Sigma Classification' (1 = Functional, 0 = Non-functional).
# - Natural and standard curve control rows were excluded from these vectors.
# - Suffix `_batch1/_batch2/_batch3` maps to increasing sequence-divergence bins
#   used throughout the analysis (see `batch_labels` in the plotting cell).
# Data arrays below are literal numeric vectors extracted for reproducible plotting.

norm_re_meDCA_batch1 = [0.8519852003028812, 0.9726443343199904, 0.7628738415999637, -0.0368153477944986, 0.9102989961134124, 1.007581653568934, 0.1644615837742373, 0.9466250275317704, -0.0078176843009224, 0.0730505021531609, -0.021653734679891, 0.502452619058682, -0.1081675950209273, 0.0474407334334844, -0.062403654560387, 0.3820824935295734, 1.088155550206362, -0.0463326768790902, -0.1091202391921531, -0.0128372297146375, 0.2772145501355811, 0.099272243136635, 0.8776811790954094, -0.1540113469925735, 0.1051389678460794, 0.6278380002970435, -0.1765620745504957, 0.1783912766044543, -0.0309473736784692, 0.4783747985649657, 0.0288557313835037, -0.1126819616767078, 0.2019935666739184, -0.1260818450737099, 0.1212939820212404, 0.8541754957241278, 0.0688955437540156, 0.0309177064312866, -0.0898009157922466, -0.0524972853146513, 0.1957450014779508, 0.0604256596181385, 0.4959371756733034, -0.0760548085056009, -0.1019112262438952, -0.0001395848700619, 1.0422584683933278, 0.7688845660053445, -0.1417509224475956, 0.948702607626226, 0.0621348837583055, -0.0119495133655666, 0.6912140830496531, 0.1574129500428282, 0.6848315692993474, -0.0135936488049727, -0.0591792454100232, 0.0089973373763917, 0.0079869462236588, 0.8294213739947689, 0.0716516886699111, -0.0642506302717261, 1.0111394698109684, 1.0225094779032156, -0.062403654560387, 0.9903042220835068, -0.1542861363612811, -0.0507754393951845, 0.657795023739427, -0.1715610298373315, 1.0598183399829733, 1.0897688374274102, 0.0448911392901601, -0.1028413159965807, -0.0648608820419165, -0.0996939621605186, -0.0326681278941483, -0.0752959523652489, -0.0474637742871198, -0.0774921442061018, -0.0943988048329546, -0.0911037094234372, 0.8878820242394274, 0.4942816499684161, 0.1540181288146157, 0.5756239829362811, -0.0608007872861393, 0.0109546656977125, 0.0358512122519257, 0.1738060836431202, 0.1702279747597333, -0.1219879978808329, 0.8279675374746134, 0.525631645415585, -0.0726421493061959, -0.0978531903904772, 0.4246685898927199, 0.2023490733040891, 0.83077564250403, -0.0448770665086861, 0.939074210543619, 0.2015209659820009, -0.0939558741913921, 1.0427495829059878, 0.2766685895642392, 1.080748136204377, 0.0552585550696191, -0.0691437051905602, -0.0629338676589076, -0.0586345058886487, 0.7838272633086932, -0.1104430806374686, 0.906566712001162, 0.6745096248779995, -0.0019674458676235, 0.1393120555071469, -0.0046362710418195, 0.0209003793310005, 0.0161497212207228, -0.1905697179680428, -0.0184126388551094, -0.0731348251128227, 0.0284469769926734, 0.0355645370320575, -0.243720956077816, 0.7806664192439531, -0.1725275668469166, 0.8317182842775336, -0.1395304424714333, 1.0108817058732569, 0.3267117879552398, 1.1104509770193889, -0.069648899102906, 0.2551184956293614, -0.0613371300309409, -0.0634620685586668, 0.9762899350448244, -0.0631982186786244, -0.0220092413100617, 0.4268962534610651, 0.5235416712588864, 0.0085754492993694, 0.7777583802176385, 0.4437071309615423, 0.0370762126997358, 1.130374355307383, -0.0966123432113777, -0.0805563662113503, 0.9497067914558878, 0.0779144567099177, 0.042086966847301, -0.0884858121176569, -0.0329309811734836, 0.9671352918178144, 0.0546413444348968, -0.1822904867735, 0.6868579866732104, 0.0462760340004558, 0.0748698845317294, -0.0127830421453176, 0.0248212236799246, 0.2150957637976255, -0.1422506744029881, -0.1502193286413027, 0.0365704943181211, 1.064883858604463, 0.8202159853518998, 0.7761416652778087, -0.059721861185183, 0.1356428943552696, 0.9812523179852546, -0.0147206247603235, 0.3871819634328719, 0.4133559389133735, -0.0876021217201825, 0.0006938354049037, 1.0435525138003232, -0.0359614753614857, -0.0713197567449494, -0.0170527257357697, 0.0417123212104801, -0.0078176843009224, -0.0932482937653024, 0.0072339375156002, 0.0931838431269365, 0.7206622226532595, -0.1351903786186969, 1.016215070153666, -0.0932009935881131, -0.2189791209555675, -0.2223864091105206, -0.1222474215720455, 0.3225977710978048, -0.1932511884101589, 0.2112920487642238, -0.0665898579411519, 0.0202575108899489, -0.0147206247603235, 0.8481922722959241, 0.7528909491360792, -0.081021854170463, 0.1427487923417716, -0.1561946195264988, -0.1820658204292524, 1.0472997717299342, 0.0632794193709382, 0.5819903682382216, 0.1469539693126469, 0.0097599736409195, 1.060716472776378, 0.6020674816392904, -0.0659994826337198, 0.6140882546734578, -0.0250991532205373, -0.1300587090531103, 0.8814098538800557, -0.1822904867735, 0.0907147018481657, -0.1404431065289079, -0.1858366854856032, -0.0855180926436032, 0.087870954952102, -0.1620278102513448, -0.0360470987231774, 0.2214209725015898, -0.0653471642387581, -0.0277476748291094, -0.0665898579411519, 0.0688955437540156, 0.8249579575695072, 0.0314380089812981, 0.9172230275356577, -0.0169475645898225, -0.0798552021589585, 1.0691913756991789, -0.078914787972783, -0.1422506744029881, 0.0519576706323493, -0.2181141040627728, 0.7553293790691772, 0.0111047258416719, 0.6667629709753211, -0.0405942433600424, -0.0346707634526333, 0.3141878642124467, 0.5465880678884609, 0.042086966847301, 0.7692068561512525, 0.997244693671347]
functional_meDCA_batch1 = [1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1]

norm_re_meDCA_batch2 = [-0.0884858121176569, -0.2167187964365249, -0.0368153477944986, -0.0092518782283843, -0.069648899102906, 0.0428392971658311, 1.089792223257935, -0.0748455848259133, -0.1081675950209273, -0.048885749781346, -0.0483284980347714, -0.1226787227230162, -0.0248210696714046, 0.1854652404743228, 0.7950780434132272, -0.1537360140243649, 0.0180305710604501, 0.0292656897654416, 0.1608926505357784, -0.1625461749847637, 0.1232966175797254, -0.0663318768292173, -0.1035018809291787, -0.1376869616130124, 0.5798268715421655, -0.0711535707311273, -0.0430576841301175, 0.2157925119416408, 0.022195093504955, 0.0351356283103748, 0.0959183273517851, -0.0174376382301076, -0.0430576841301175, -0.0496155929327466, -0.2970392373556671, 0.0383851324098777, -0.0086018033892137, -0.2655876659888271, -0.0298609976971643, 0.0773333253018044, -0.0830974965006294, -0.1005043328343685, 0.6721590006255551, 0.3890827610788321, -0.0561255737731645, -0.1201759041708025, -0.1615075077013334, 0.180674080399089, -0.0571713568549893, -0.1696075834541385, -0.0116948366796116, -0.1111933831016395, -0.0433016469796806, 0.0417123212104801, 0.0124625977832117, -0.0316117222908792, -0.1572735456274551, 0.0161497212207228, -0.007029122587863, -0.1159741701805466, 0.0381661224983769, 0.9120961095559104, -0.0444244457194131, -0.0393459769656081, -0.0604779867470696, -0.1126325980407796, -0.0038294237385031, -0.1975817877244511, -0.067617026957812, -0.049848336749592, -0.0292724946214537, 0.1653683020953314, -0.1419510386144071, -0.1066296703116638, -0.2163677728707862, 0.1946180255225998, -0.0768095495566916, -0.0222929942524634, -0.0657626295324582, -0.1060097104000096, -0.02893508652884, -0.1286949235846305, 0.0106550299091315, -0.0628279865050452, -0.1948397210761449, -0.1298407281627585, 1.0180971893292017, -0.15807728666784, -0.0526601651109349, -0.0446760834123314, -0.2635993354583664, -0.0853680324996436, -0.2015473200718453, 1.073012580871037, -0.0175191496643786, -0.1267525970709081, -0.0405942433600424, -0.0022015599178448, -0.02893508652884, -0.0378561726952003, -0.0541980898201985, 0.2756069861902011, -0.149160051562665, 0.022195093504955, -0.1163352099939985, 0.0886026617746488, 0.0417123212104801, -0.1041088630078205, 0.0048513269196482, -0.0292050784079582, -0.1707504231776765, 0.0091664513340632, -0.1651094292004855, -0.0602623697093068, -0.0405251881514613, -0.1628046351705331, -0.1422506744029881, -0.0229528433653161, -0.1175319565766137, 0.0419165483491634, -0.1089302312854552, -0.0234222561881548, -0.1000321902007005, -0.0914737072577482, -0.1882233199129974, 0.0391176644694907, -0.1347172424008852, -0.0541547380771164, -0.1559235760717253, -0.0856893920745067, -0.1156121913808271, -0.1058542879717535, -0.1892947707012867, 0.022195093504955, -0.1829623180218646, -0.1269012150989535, -0.0943988048329546, -0.1266410293133988, -0.0224818420019791, 0.2388354734035152, -0.0823323446557837, 0.0283449756909002, -0.0712248176075437, -0.1639191160318315, 0.0143308240893002, 0.0103560378069507, -0.2034885059325514, -0.065728054753982, 0.001784796679895, -0.0243073515082825, 0.3186839647893889, -0.0709396110117584, -0.1483984585814107, -0.0524972853146513, -0.0361754350808549, -0.139632145632729, 0.0760014417405176, -0.1039755538879213, -0.1442319287846832, -0.0241234202405466, -0.0686366708591697, -0.1106936311462467, -0.1930019837418983, -0.0359002835512936, 0.0383851324098777, -0.1757644748316509, -0.0445150877089662, -0.2257122727396694, -0.06062154608939, -0.1829623180218646, -0.0800558954454127, -0.0845775513562864, -0.025073895844542, -0.0256536590139195, 0.937225159147431, -0.1693614635490833, -0.0969658086153907, -0.0561696727469627, -0.0845290083911812, -0.0217249086881103, 0.4677509615328025, -0.0530438073652109, 0.8780405106901983, -0.0912430713756435, -0.1794939286705563, -0.0159622098898434, -0.0173435272183131, -0.1767957954619086, -0.087036837862012, -0.1492549583307749, -0.0530665326114103, -0.0532557655026141, 0.0486152616698813, -0.093118694103091, -0.2094505347309536, -0.1487797759981197, 0.1098480201820021, -0.0901495215350209, -0.0548467507894404, -0.057208123668052, -0.1691149076475114, -0.0851426384908057, -0.0372788978317899, -0.0210932402484407, -0.0436667913169006, -0.0911037094234372, -0.1456995779206084, -0.0699511375930988, -0.0409045685054152, -0.0978531903904772, -0.0447565114816253, -0.033367969821443, 0.0167113121778457, -0.0611228408892411, -0.1070585790333465, -0.114154787801978, -0.1561946195264988, -0.1023105974506028, -0.1284815729701431, 0.0272278675628456, -0.1089302312854552, 0.9585416182202372, -0.0937944583773704, -0.0760869110936893, -0.0652951437671974, 0.8867106245445702, -0.0568128555957247, 0.2808164716347335, -0.050731008183302, -0.1455047832491471, 0.0295396677476932, -0.1250696168299736, -0.0287848644203114, -0.1300587090531103, -0.0478574966714344, -0.0519257002400949, 0.0281751394647743, -0.0719825235689635, -0.1128463807980616, -0.1401395496582164, -0.0786786882214418, -0.1083586462108988, -0.0436667913169006, -0.0155200569986387, -0.0898493855019618, 0.0066233728393539, -0.0422925322852718, -0.0154669041356518, -0.167925321174624, -0.105594864280541]
functional_meDCA_batch2 = [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

norm_re_meDCA_batch3 = [-0.0997616735556949, -0.0812540156422085, -0.1328618247304829, -0.0307252175737769, -0.1467451038576518, -0.1849585704589727, 0.0288557313835037, -0.0938429027809363, -0.1594066260002234, -0.1191903002642139, -0.1466694826821757, -0.0279179265568094, -0.0643030425374707, -0.1145205719022419, -0.0988788455113076, -0.0103776586780656, -0.0038294237385031, 0.0019533984813002, 0.0027994847953192, -0.2038735152238175, -0.1994464383707353, 0.031960263890667, -0.0665898579411519, 0.0148480879659567, -0.1526292028632017, -0.1171196973881794, -0.0765355715744399, -0.1169747551516595, -0.1029736800294195, -0.0959181733432651, -0.0970271893308463, -0.0001395848700619, -0.2620892218731918, -0.2057827463972691, -0.1871435567779685, -0.1658692897050133, -0.0054384679025793, -0.0258610304504047, 0.0497997860114314, -0.1607233886130419, -0.0807893047929871, -0.0389273878680158, -0.0674707518313265, -0.0292724946214537, -0.0651216016374318, -0.1438132656200769, 0.0486152616698813, -0.0361754350808549, -0.1419510386144071, -0.0215350303912116, -0.0634620685586668, -0.1066296703116638, 0.0235019647973202, -0.007029122587863, 0.0405943973685623, -0.1725810670474752, -0.1800273349573157, -0.0959181733432651, -0.0813865056760496, 0.0032244645760132, -0.0176820294606624, -0.0086018033892137, 0.0207986761697051, 0.0281751394647743, 0.0926433346028125, -0.1462338617630505, -0.1012698390717813, -0.1858366854856032, -0.1599348268999823, -0.1576313370071149, 0.1404482313679802, -0.0357165465344581, -0.0368153477944986, -0.0349791632465321, 0.0521995139078649, -0.0774921442061018, -0.0018736898721349, -0.0285968584886578, -0.2526673778678146, -0.1609852527919645, -0.0599022640231946, 0.0293683683167897, 0.0030421714654218, -0.0652951437671974, -0.2065391654876042, -0.0314222317646182, 0.0609401917015124, -0.0234222561881548, -0.0869356522741022, 0.1891160929441283, -0.1191903002642139, -0.0902365367113718, 0.0428392971658311, -0.187576467764753, -0.0647448497133622, -0.1058542879717535, -0.0139703222961528, -0.1328618247304829, -0.0577218418311741, -0.0536334611754846, 0.0025572205565356, -0.0132159531321022, -0.1437392529780564, 0.0047221967696341, -0.1050745617305295, -0.3527484811662705, 0.0079869462236588, -0.0490320249078313, 0.0422120731658727, -0.1366521555818896, -0.0774921442061018, 0.0274978594419639, -0.0454782802730913, 0.2302390296139241, -0.1087611173277837, -0.0205817337188781, -0.1084949509618183, -0.1310354202195517, -0.1477286289005938, -0.0576171552748736, -0.0861540494255744, -0.0035802190702425, -0.0710981312681199, -0.0231094902811653, 0.0474407334334844, -0.1071808820025871, -0.0659583912071157, -0.066884109150203, -0.0757333747136801, -0.0857179214934933, 0.0309177064312866, -0.0736957559276538, -0.0318954752332808, -0.0209189374453968, -0.1202779054725757, -0.116623369041114, -0.1367905750562922, 0.031611876299399, 0.0260193438767204, -0.040718456460692, -0.1008789784711893, 0.0011124245024962, -0.0601724305979123, 0.0170869690051957, -0.0727517836324947, -0.132005324981119, -0.2600499067071576, -0.043514764245488, -0.1184819996513775, 0.1145033294017659, -0.0858177284321842, -0.09884477850097, -0.1620278102513448, -0.1214676953308212, 0.0648610360504365, 0.0538226316621284, 0.098145267181284, -0.0728394289565747, -0.0200066057515767, -0.0407403649168131, -0.2378225831913177, -0.234004988676535, 0.0515956918326296, -0.1110864443564322, -0.2072914958061344, -0.0387176202773811, -0.0487855781501419, -0.1700985214236222, -0.0351023320776144, -0.0574599776522518, -0.0474637742871198, 0.0578811505397583, -0.1499444754311483, -0.0748455848259133, -0.137377326846731, 0.0225206627061332, 0.0825860532120154, 0.0243250350259068, 0.047051416340366, -0.1472481977365003, 0.0379162467220322, 0.0011124245024962, -0.0032889152143791, -0.1062425191022538, 0.114819290422908, -0.0285968584886578, 0.0368592512060116, -0.0536334611754846, -0.0188491706401677, -0.1744407347033313, -0.0091221059392252, -0.114154787801978, -0.0176820294606624, 0.5163823118888988, -0.0915352780484182, -0.0042334320629389, -0.0569873769603344, -0.0711535707311273, 0.0576651989525399, -0.0778095463277885, -0.1758585476774358, -0.0348867150398518, -0.0074802762083085, -0.0005544309895305, -0.0569873769603344, -0.0511215169130072, 0.0213844868452999, -0.1202460380952091, -0.0975784010217696, -0.004904184060137, -0.1170544920601938, 0.0076350351131939, -0.1351903786186969, -0.0945193598600155, 0.0268238592384096, 0.0486152616698813, -0.0750882714960157, -0.0080795484798449, 0.0045072457307087, -0.1705877337727819, -0.0201711706776834, 0.1404482313679802, -0.1089302312854552, -0.0436667913169006, 0.0931421909434625, -0.0736257630823064, -0.1584507833038181, -0.0871581635445808, 0.3282329044077304, -0.0645124944506487, 0.0288557313835037, -0.0248210696714046, -0.0806340557305543, 0.0661929196117234, -0.2625944157855376, -0.1151280838665276, -0.0610692170126234, -0.0330950127423108, 0.0019533984813002, -0.0097143654288475, -0.0961962568923978, -0.0518620456448671, -0.0940365119514923, -0.0448770665086861, -0.0945193598600155, 0.0405943973685623, -0.0205817337188781, 0.0820517693695731, -0.084916878879198, -0.1578098883722567]
functional_meDCA_batch3 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

norm_re_eaDCA_batch1 = [-0.0892434678643546, 0.0511256445791936, 1.076495211238267, 0.3097952371433661, 0.4839849163878942, 0.6126571777590356, -0.0638458750525968, 0.1758935600975149, 1.069980412138149, 0.015650414525504, -0.0163199286988286, 0.0352386957429747, -0.0314663649006483, 1.1046903130611556, 1.0608897622982991, 0.0659880530382574, 0.9582925115020215, 0.003857449208562, 0.0317951143448593, 0.7239417015988835, -0.0935969687303435, -0.0874668343581029, 0.790777826478779, 0.0889012377674325, 0.1158534787787101, 0.9380976701581376, 0.0747466038846997, 0.0659880530382574, 0.1461668957419951, 0.648432406338663, 1.0426651689213384, 0.0219056880826045, 0.1902492606976479, 0.0808767382569404, 1.1405469863690414, 0.7180880291866317, 0.8201025730604887, 0.4188198980876995, 0.196770794831464, -0.0070924167416067, 0.3904100201791015, 0.0659880530382574, -0.1532030850937175, 0.1485800625180992, -0.0231314022253828, 0.15605632200425, 0.2863552972603822, 0.9331918021707428, 0.0508040643492152, 1.0136822316883172, -0.0570067596168227, 0.003857449208562, 0.2138702200031541, -0.0544421319967725, 0.7638071377095663, 0.1039773815333249, 0.1522726007920088, 1.1102058082501929, 0.0527437475067576, 1.0481991109334263, 0.7440569358914989, 0.082033218639534, 0.838725773194156, 0.0793210606986275, 0.7072829958780059, 0.2822450132333142, -0.1210330526549729, 0.2181869258339453, 0.4488144118696062, -0.0402249157470906, 0.0577031746658874, -0.0221766768730481, -0.0412040431893479, 0.1858916327345947, 0.2572448103824758, 0.1809595726700477, -0.0173117310179154, 0.1267063618627652, 0.1034070671694118, 0.6456454992263656, 0.9093523332777969, 0.3904100201791015, 0.3213654801487938, -0.0314663649006483, 0.0725095871720733, -0.0425715607407979, 0.0577031746658874, -0.0238773927037557, 0.9798792557963464, -0.0387994829253438, 0.8494548340138917, -0.028869134530827, -0.0185425241678938, -0.0017152712229015, 0.0010457570495767, 0.2053335208290894, -0.0402249157470906, 0.0177072631711135, 0.0146322518204335, 0.9910667126822368, 0.0681312057826533, 0.0692141284760137, 0.003857449208562, -0.0610848467801186, -0.0744178544404887, 0.0642948543369066, 1.154901192924527, 0.0201742241985492, 0.4093872021884551, 0.0609635726311353, 0.0390857494311513, 0.8788217306487173, 0.2685245643466987, 0.4952584837472211, 0.1434547378010888, 0.2087334399036881, 0.1750652720086059, 0.0674134858600041, -0.0702761719563612, -0.0164103582814736, 0.0231879794428964, 0.1722010218236054, 0.1639161434512355, 0.0939257181745546, 0.2841970513937535, 1.0196049852269455, 0.7244736582797476, 0.030664238929047, 1.044030833108001, 0.0136208097102347, 0.4359525299153709, 0.8445399507545873, 0.2476646333136707, 0.7346334691450667, 0.0081741550393532, -0.0105751330967334, 0.9037256163966016, 0.1952973899856482, 0.9230941959409392, 0.0067216993935625, 0.5939749442292969, -0.0284319504301486, 0.2984918342982653, 1.0619958623794414, 0.0617896281267662, 0.0284272222164206, 0.6409914135299745, 0.07575141353993, 0.4307964858632169, -0.068466313239216, -0.0291962059961883, 0.6868549053542748, -0.0589024488252775, 0.8197893985800953, 0.062444937422063, 0.1281186568679526, 0.5103202608732161, 0.1008920506565272, 0.018505769008006, 0.1779840826084054, 0.050354982820576, 0.8474554944413328, 0.822859720004091, 0.0010457570495767, 0.0524187570915216, -0.1106403979491559, -0.125976478882292, 0.0889012377674325, -0.1185002193961414, 1.1095130927592505, -0.1952354807017637, -0.0189506014352916, -0.1079282400082495, -0.1323290123114899, 0.5026328934965915, 0.4746952283602941, 0.1175466774800607, 1.0645657726050253, 0.6565617118987601, 0.0882996157800553, 1.0019922522103295, 0.0423670937327512, 0.0206340382670476, 0.0498433532189019, -0.1210330526549729, 0.0703047588690486, 0.0219056880826045, 0.2922638045715048, 0.1570481243233368, 0.9625990340516432, 0.7510651983021348, -0.0601533367786872, 0.2889859699968454, 1.0218673571572734, 0.015650414525504, 0.0219056880826045, 0.6772317396687354, 0.0206340382670476, 0.2076149464460534, 0.8372259283649522, 0.7666561718751564, 0.4892353802897647, -0.007970544780002, 0.6925678206018716, 0.2418078851394514, 1.137407428919338, -0.1649583361787818, 0.6601645570901878, 0.9428686866730794, 0.0939257181745546, -0.0103597670411196, 0.0219056880826045, 0.2181869258339453, 0.2841970513937535, -0.128389645658396, 0.9469233352139942, 0.0423670937327512, 0.0326487818638725, -0.0738492715122548, 0.9725767581846348, 0.5325091263591982, 0.0747466038846997, 0.0840362919122998, -0.0524900938363332, -0.0205721289831631, 0.1722010218236054, 0.0965011220471743, -0.0067985855425701, 0.0564078759694523, 0.4321529057480696, 0.1044976975624465, -0.1526931580895393, -0.2099807833673215]
functional_eaDCA_batch1 = [0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]

norm_re_eaDCA_batch2 = [0.015650414525504, 0.0126160000550044, 0.0939257181745546, -0.0444233406585818, 1.1070596997987143, 0.1722010218236054, 0.7225004299564058, 0.0856408398021847, 0.0366380928967725, 0.0498433532189019, 0.0010457570495767, 0.0747466038846997, -0.1361696375003748, -0.0070924167416067, 0.0593247022137591, 0.1666283013921418, -0.0021706442871413, -0.008843669212678, 0.559782005535899, -0.0753437359483437, 0.0387614468268319, 0.7103812296287574, -0.0444233406585818, -0.0864208717757165, -0.008843669212678, 0.0219056880826045, -0.0122872506107933, -0.0601533367786872, 1.092888295240387, 0.0939257181745546, -0.04984808143263, 0.376767137947316, 0.003857449208562, -0.0744178544404887, 0.0747466038846997, -0.0122872506107933, -0.0209748438623837, 0.0703047588690486, 1.0279458205784864, 0.3169236351331426, -0.0794423348476108, 0.063176360879272, 0.0084663056013399, -0.0233691570028632, 0.1180117246453389, 0.0738478744852428, 0.1158534787787101, 0.0448188728117796, 0.0531626076947043, -0.0727057369264286, 0.2230518716890779, 0.7750028949014031, 0.0959825668201641, -0.1097416685496989, -0.0114335830917801, 0.0747466038846997, 1.081622209159981, 0.0889012377674325, 0.0609635726311353, 0.0352386957429747, 1.05212074364614, 0.1281186568679526, 0.0035031532742041, 0.1158534787787101, 0.0543445103790307, -0.1606551007821204, 0.0808767382569404, 0.1485800625180992, 0.0840362919122998, 0.1902492606976479, -0.0744178544404887, -0.1271878126477318, 0.0659880530382574, -0.0920872725447221, 0.0352386957429747, -0.1889157015982065, 0.1081902358363561, 0.1281186568679526, -0.0423385068200637, -0.0695748758885755, 0.2572448103824758, 0.3192050607501067, 0.0885214781595637, 0.0335072318589192, 0.2728412701774899, -0.0485097941194605, 0.0677001705523173, 0.1779840826084054, -0.0323224169781903, 0.0735223662941286, -0.1654290982183137, -0.0444233406585818, 0.6437385509949205, -0.0787345602712797, 0.018505769008006, 0.0939257181745546, -0.0656593035940462, -0.1464378845324387, -0.0056057377709429, 0.0560986267760026, -0.0359082099162994, 0.1158534787787101, -0.0091915307090521, 0.9690813044009278, -0.0874668343581029, -0.0122872506107933, 0.0617896281267662, 0.0840362919122998, -0.0843072807027433, -0.042687954915285, -0.008843669212678, -0.0444233406585818, -0.0965724587919859, -0.0920872725447221, -0.0359082099162994, 0.079985846658224, 0.1226314697478613, -0.0122872506107933, 0.0004412871400975, -0.0276638639931393, 1.0817310780266172, -0.0257266330920579, -0.1222839406083825, -0.0337033816132747, 0.7607215211743522, -0.1331048768720684, 0.015650414525504, -0.1307653974853839, 0.296462229482996, 0.2107106663477946, 0.0840362919122998, 0.8753931595776301, 0.722073311412867, 1.1018195566311435, 0.1414516645283228, -0.0474983520092617, 0.2107106663477946, 0.0324776674704965, 0.9299525798990076, 0.0573743485239396, -0.0030773506211035, 0.015650414525504, -0.0438305626530098, 0.2107106663477946, -0.0139804493121439, 0.1341966800601826, -0.0359082099162994, 0.1677847817747353, 0.7465600306249722, 0.0537228749490147, 1.036907954965198, 0.1245130099620335, 0.7559380785481221, -0.0221766768730481, 0.0414576968597723, 0.015650414525504, -0.0890225119164157, -0.1354091747815364, -0.0426380825231947, 0.0392713738310101, 1.0761588905857231, -0.0017152712229015, 0.1604981670199011, -0.0126273904249192, 0.0212685440322279, 0.015650414525504, -0.0965724587919859, 1.0991954290866688, -0.0017152712229015, -0.0874668343581029, 0.1722010218236054, -0.0245524287000358, 0.7787371020966485, 0.036989948214046, -0.0237645970245036, 0.0179663214295108, 0.0453139715408618, 1.0957651534935022, -0.0053209181288207, 0.015650414525504, 0.0688523032232578, 0.0352386957429747, 0.0659880530382574, 0.0373969416096035, -0.0477011752332412, 0.0342468934238878, -0.0221766768730481, -0.0052318033119183, -0.090562554259844, 0.009640509993362, 1.0000334630307754, -0.0245524287000358, 0.1255858236091211, -0.0522935153317016, 0.1746528009026339, -0.0819678013160586, -0.0044274291638078, -0.0464801893041912, 0.0021398484172888, -0.0156551427392321, -0.0260453151965478, 1.1868281993605108, 0.978217643179362, 0.0819976863672947, 0.334433269152674, -0.0333858003113166, -0.0874668343581029, 0.0441463058651252, 0.0249145658605454, 0.0361118201756507, 0.0548269769604455, -0.0053209181288207, -0.0187875009160668, -0.0485097941194605, -0.1375361922245411, -0.0438701526088805, -0.1237492169472318, -0.2181916540476734, 0.0179440992667071, 1.085247499549633, 0.0450662222171348, -0.0554089044361328, -0.0848866119158582, 0.1170777163692812, -0.0097118467381736, 0.0414576968597723, -0.0138508654368994, -0.0747579942546145, 0.0710361823262575, 0.1410541729348848, 0.0089055784965622, 0.1281186568679526, 0.0256585186741697, -0.0464801893041912, 0.0372014498671474, 0.011256337865831, -0.1436458969865254, 0.0647767192728176, -0.0806147424288338]
functional_eaDCA_batch2 = [0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

norm_re_eaDCA_batch3 = [0.9312429811254284, -0.0070924167416067, 0.0352386957429747, 1.0601784362482798, 0.0103354450771768, 0.0438334486867601, -0.1243463260256261, 0.0399539269566471, 0.0991205520437413, -0.0562099147460031, -0.0341468925548605, -0.0195762981941886, 0.0053359313378646, -0.101644460651914, -0.0748429113658731, 0.0530310772803237, 0.1425269877015911, 0.0545107066245471, -0.0507338233401653, -0.0205721289831631, 0.0240488408270006, -0.0221766768730481, -0.0328373070724057, -0.146104986458111, -0.1193491574309538, -0.1032984943886183, 0.0955054530252233, -0.0575718585533474, -0.0420383442885401, -0.0402249157470906, 0.0106568309676334, 0.0146138022480383, -0.0097118467381736, -0.2099807833673215, -0.037401962671448, -0.0097118467381736, 0.0973692995726699, -0.0245524287000358, -0.0139804493121439, 0.0503982105180202, 0.0445283837705749, -0.0874668343581029, -0.0284319504301486, -0.0873178475468084, 0.0655158402659568, -0.0601533367786872, -0.0662590418287009, 0.0703047588690486, -0.1442947317880427, -0.1596821900639384, 0.0560986267760026, 0.0423670937327512, -0.1490753493578789, -0.0444233406585818, 0.0317951143448593, -0.0638458750525968, 0.063176360879272, -0.2188564400072699, -0.1799910188585588, 0.1744881228391273, 0.1188289688403524, -0.1299775658098517, -0.0291962059961883, -0.0004942852938514, -0.0617467610851512, 0.0524187570915216, 0.1171687909177837, 0.2053335208290894, 0.0449955102069529, -0.0515046697113135, -0.1402860319370244, -0.0435928078755293, 0.0010457570495767, -0.0194156486005699, -0.0686347936556885, 0.0514648807667735, -0.0027200808781315, -0.0149677782650057, -0.0392081042964092, -0.1682236970733216, 0.1324353626987439, -0.0877792359410185, -0.1952354807017637, -0.0021524553235797, -0.1274289345280828, -0.0830249893424516, -0.1106403979491559, -0.0163821047692068, -0.0017152712229015, -0.0838505503397331, 0.0582417795203985, -0.153101235356937, -0.033258583265118, -0.0677058505203774, -0.0163199286988286, -0.0127971776149713, 0.0120162618203498, -0.1439346072430968, -0.0345434346301435, -0.1218681115245801, 0.020317767931149, 0.0094263479412476, 0.0136208097102347, 0.0387614468268319, -0.072796326892617, -0.0402249157470906, 0.0016661341048355, -0.0030517627570807, -0.2609357316901566, -0.0898164666633498, -0.0097118467381736, -0.0017152712229015, -0.0392394916220709, 0.0265354337022357, -0.11954822588811, 0.5020060846986044, -0.187294174050335, 0.0577031746658874, 0.015650414525504, 0.1158534787787101, 0.0423670937327512, 0.0355068199117259, -0.0658231014967033, -0.0951282169664294, -0.0422385179595209, 0.0962128191900765, -0.0117526722149253, 0.0719979575703992, 0.1546038663584592, -0.0620107282879736, -0.1547227629048086, -0.0166039564415845, 0.0073010306066775, 0.1178307052228166, 0.0029398840516002, -0.0268919080867205, -0.0452736246187024, 0.0195107713599201, -0.0490847776209223, 0.009640509993362, 0.15605632200425, 0.0840362919122998, -0.1274289345280828, 0.0549233628614006, 0.0276604169965013, -0.3075178735085802, -0.183665237696336, -0.1404705387905564, -0.1921081696395469, -0.0197635100969439, -0.1862035436573002, -0.0605680404779371, -0.0837428862402081, 0.0101772283121194, 0.0352386957429747, -0.0874668343581029, 0.0569717512086785, -0.0122872506107933, 0.1180117246453389, -0.0300733287308213, 0.5718748229609294, 0.1014019776607054, -0.1037678145819734, 0.0753486986607094, 0.015650414525504, -0.0343078515227538, -0.1185002193961414, 0.0725984436102517, -0.0461392927824026, -0.0341468925548605, 0.0175480601195513, -0.0060319770536925, 0.0692141284760137, -0.0274712392998354, 0.0317031101851787, 0.0231879794428964, 0.1067025258654714, -0.0138119688956713, -0.0548295732230177, 0.0498433532189019, 0.0742173013779434, 0.101166415856675, -0.000340975702929, 0.0757154137927082, -0.0403950800325896, -0.1559277462310198, -0.1084745255185525, -0.0139804493121439, 0.0006482654561389, -0.0681625808833878, -0.0341468925548605, 0.0801426869209901, 0.1124001963313468, 0.0187461344272451, 0.8174137860070895, 0.0927948427587423, 0.0983356030433917, -0.0971606857075993, -0.0661093271872407, 0.1870897070422885, 0.0219056880826045, -0.0716061622815031, -0.0056445358274472, -0.0295225808231835, -0.046253009242794, 0.07295438552023, 0.0762563000703211, -0.0642507178860919, 0.00883894099895, 0.1902492606976479, -0.0291962059961883, -0.0282786408392264, 0.0167689079831386, -0.0994912693917855, -0.0674515219585161, 0.0511256445791936, -0.0153216650812926, -0.1046598174862538, -0.0047258380669666, 0.1604981670199011, 0.1697878550475012, -0.149857817049676, 0.1485800625180992, 0.0685634569108771, 0.0200922595411551, 0.030664238929047, 0.0444662077001966, 0.0423670937327512, 0.0873555463881022, 0.0076884718329228, 0.0204933930774171, 0.1020845307863424, -0.1159248155235217, 0.0805927105141845, 0.1139014406182708, -0.0077127937968655, 0.0236569405536759, -0.1899872648695413, -0.018016251446772, -0.0337033816132747, -0.0485097941194605, 0.262269290789598, 0.0644001328868017, 0.0577031746658874, 0.0033423800736987, -0.0143168554260626, 0.0025656993177687]
functional_eaDCA_batch3 = [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

norm_re_edDCA_batch1 = [0.036989948214046, 0.9485846886368168, 0.9593115337085536, -0.0032709487812143, 0.6430388009753374, 1.0158324329625166, 0.9416839633635772, 0.9998159502124412, 0.9604758743207172, -0.0063357094095207, 0.0892311257814865, 0.8284364702827152, 0.0913928849157232, -0.0189506014352916, 0.0617896281267662, 0.9756792544063412, -0.0777857465689274, 1.0188227569381714, 0.0559213764111318, -0.0314663649006483, -0.0451524439691722, 0.8657803271149246, -0.0897944678228347, 0.9375237634073252, 0.0577031746658874, -0.1037678145819734, 0.0423670937327512, -0.0097118467381736, -0.1075368402466742, -0.1596821900639384, 0.0141256962406259, 0.0726805106960362, 1.0636799171870943, 0.1023241444760512, -0.0342575700052083, 0.9224844023112236, 0.9884699762417212, 0.0126160000550044, 0.7241689375507834, -0.0221766768730481, 0.1089395425780978, 0.0600249300431434, 0.2695923939178263, 0.0720660762304873, 0.8844628433702449, -0.0626248891235466, 0.9870140888281176, -0.0485097941194605, 0.0295416483891982, 0.003857449208562, 0.09673741033354, -0.0727057369264286, 0.2713492143445438, -0.0485097941194605, -0.081081205264987, -0.1635373097041288, -0.0017152712229015, 0.015650414525504, -0.0141907896654803, -0.0017152712229015, 0.1119739570485972, -0.0323650943001052, 0.0677001705523173, 0.1722010218236054, -0.0061286844872243, 0.0128472950775531, -0.0521154410253799, 0.0324776674704965, 1.0844972076298622, 0.8383192802267402, 0.0889012377674325, -0.0303354894848358, 0.3013271753381286, -0.0084077288806803, 0.7081203363405636, -0.0320286881861865, -0.0556015291294366, -0.0070924167416067, 0.0352386957429747, -0.1936798031434509, -0.1331048768720684, 0.1044976975624465, -0.0399205801488487, 0.0747466038846997, -0.1421211787016475, 0.0747466038846997, 0.7959715995417784, -0.0433844694024501, 0.1313447323057091, 0.1067025258654714, 0.0237408348472277, 0.959102303458762, -0.0880910019149846, -0.0122872506107933, 0.0187461344272451, 1.048866667105596, 0.031309431138429, 0.0715034579937039, -0.0291962059961883, -0.0306447386782855, 0.112213570738306, 0.0004895570801233, 0.9658323784929886, -0.0764474592557579, 1.0277702220136145, 0.030664238929047, 0.015650414525504, 0.0317951143448593, -0.085895200854199, 0.2071050194418753, 0.0577031746658874, 0.9933197417802152, 0.0019772670510081, -0.101644460651914, 1.0792241488977496, 0.9156918033210416, -0.0662590418287009, -0.0761110531418393, 0.0939257181745546, 0.2162833867792582, -0.0638458750525968, -0.0454552160956265, 0.0387614468268319, -0.1229055760383983, 0.0067216993935625, 0.0168812076754825, -0.1526931580895393, -0.1307653974853839, -0.0070924167416067, 0.0091501642202303, -0.0490847776209223, -0.0439047854315717, 0.0991205520437413, -0.0173117310179154, -0.0505128673922266, -0.0674515219585161, 0.1094411237897658, 0.23283665844576, -0.0721307534249666, -0.1004519805220988, -0.0173117310179154, -0.0698089980477106, 1.0750030277879743, 0.15605632200425, -0.1928960013150792, 0.0653811891822969, 0.3405445944386487, 0.9916826532829044, 0.0955054530252233, 0.1607236754098951, 0.032282341856611, -0.0323650943001052, -0.0329613321081889, 0.3729241045905972, -0.0229726938493671, 0.0955054530252233, 0.974905184251218, 1.106909363824007, -0.0794423348476108, 0.0382532111259396, -0.0796775350831122, -0.0185425241678938, 0.2320530271859634, 0.795709886543326, 0.0235102359724895, -0.0437315429898907, 0.2841970513937535, 1.0652456894426363, 1.0188464131209036, 0.0688523032232578, -0.1442947317880427, 0.0019772670510081, 0.015650414525504, 0.0067216993935625, -0.0221766768730481, -0.053956448790342, 0.9841677912210752, -0.0104521038461702, 0.7919503835841728, -0.0139804493121439, 0.009640509993362, 1.0850411687580317, 0.015650414525504, 0.0840362919122998, 0.1608452406073418, 0.044571922035776, 0.0190939959236193, -0.0702761719563612, 0.0161620441271781, 0.0717711138230574, 0.9894884919915432, -0.0853677203906575, -0.0262271221271239, 0.0808767382569404, -0.1032984943886183, 1.006972024165188, 0.2461245909702425, -0.1553933675783874, 0.8492523435067876, 0.8683116917741535, -0.039157086175963, 0.8283342724142534, -0.0419181144484412, 0.906987354781034, 0.0504338469105026, 0.0333924936106946, 0.0242451674692893, 0.9864848617059632, -0.0882466473558372, 1.069889980331844, 0.015650414525504, 0.0423670937327512, 0.0022444561742322, 0.6403350881788761, -0.0156551427392321, -0.0329613321081889, -0.0337033816132747, 0.8572348258558704, 0.0939257181745546, -0.0524900938363332, -0.1921081696395469, 0.9957402904887642, 0.0136208097102347, -0.1777665730408362, 1.070568364836178, -0.0237645970245036, -0.0189506014352916, 0.0237408348472277, 0.1639161434512355, 0.1001809917316555, 1.0270687091263246, 0.0321739351146683, -0.0080501256300612, 0.0577031746658874, 1.126308526099078, 0.5446491652635806, -0.1957505498366273, -0.0438305626530098, 0.4908399281796495, 0.5200780219696662, 0.29132544938353, 0.2442210519155554, 0.5520098208790315, 0.0902552930290975, 0.15605632200425, -0.1210330526549729, 0.1827730012114973]
functional_edDCA_batch1 = [0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0]

norm_re_edDCA_batch2 = [0.0577031746658874, -0.0674515219585161, 0.0126160000550044, 0.0284272222164206, -0.0601533367786872, 0.0840362919122998, -0.0601533367786872, -0.1005716470048262, 0.009640509993362, 0.0477679320112013, 0.0171904568689323, 0.0781039540349882, -0.0185425241678938, -0.0935969687303435, -0.0595291692218055, 0.0329343978335067, 0.0362369593605106, 0.0067216993935625, 0.1414516645283228, 1.1285264191931812, 0.1586317258768697, 0.0537228749490147, 0.0441143523829232, -0.0184062565525599, -0.0314663649006483, 0.1902492606976479, 0.121863383310852, -0.1485514756054118, 0.1001809917316555, 0.0872623673500562, 0.0638744619652842, 0.196770794831464, 0.006000601952958, -0.0017152712229015, 0.2258501687762993, 0.0577031746658874, 0.07295438552023, 1.0946802269686204, -0.1650761623463287, 0.0571999462223252, 0.0120162618203498, -0.0231314022253828, -0.0920872725447221, -0.0485097941194605, 0.030664238929047, 0.1198337784955828, 0.0136208097102347, 0.0295416483891982, -0.0919492870123498, 0.0284272222164206, 0.0659880530382574, 0.0765600324261492, -0.0148200838696248, -0.043036607906076, -0.1154913416182007, 0.0659880530382574, -0.0173117310179154, 0.262269290789598, 0.1158534787787101, -0.0885057056142345, 0.2174226702679056, 0.1044976975624465, 0.1287922092638539, 0.003857449208562, -0.0727057369264286, -0.145370054961311, 0.003857449208562, -0.0291962059961883, -0.0490847776209223, 1.0715042771830672, 0.1263052283265032, -0.1079282400082495, 0.009640509993362, -0.0665580329935031, 0.073251636677159, -0.0423385068200637, 0.1119739570485972, 0.1809595726700477, -0.0457976361785542, 0.1281186568679526, 0.1100704179939101, 0.1158534787787101, 0.2871725414553958, 0.0073010306066775, 0.0284272222164206, -0.0044274291638078, 0.015650414525504, -0.010747208267365, -0.0370712737417134, 0.1044976975624465, -0.0524900938363332, 0.1198337784955828, -0.2725792743493833, -0.008843669212678, 0.8262447820494445, -0.1235246998032635, 0.0498433532189019, -0.0561678605361872, -0.1023555195767859, 0.2343316256533006, -0.0221766768730481, 0.1008920506565272, -0.2472356988882218, -0.0444233406585818, 0.7603336761527555, 0.0498433532189019, -0.057894333851324, -0.0444233406585818, 0.1722010218236054, 0.0171904568689323, 0.0423670937327512, -0.1376793336859963, 0.1318111951418622, -0.0732785709518412, 0.0659880530382574, 0.8722551743540529, 0.0471628255646895, -0.2926898986406695, 0.0001198755417215, -0.0070924167416067, 0.2576255134934455, 0.0102310036849627, 1.0008376389341933, 0.1158534787787101, 0.0659880530382574, 0.1364680407867792, -0.0221766768730481, 0.0747466038846997, 0.0747466038846997, 0.1902492606976479, 0.0317951143448593, 0.0939257181745546, 0.0939257181745546, 0.1722010218236054, -0.0505128673922266, -0.1159248155235217, 0.1485800625180992, -0.0173117310179154, 1.045131882255371, -0.0994912693917855, 0.009640509993362, 0.003857449208562, -0.0524900938363332, -0.0359082099162994, 0.025131763520361, 0.1281186568679526, -0.0122872506107933, 0.2902069559258953, -0.069223020571302, 0.330655168176394, 0.0939257181745546, 0.1722010218236054, 0.0747466038846997, -0.0221766768730481, 0.0659880530382574, 0.003857449208562, -0.0359082099162994, 0.1281186568679526, -0.0233691570028632, 0.2220664475640582, 0.0423670937327512, 0.1722010218236054, -0.2099807833673215, 0.2107106663477946, 0.1100704179939101, 0.0057609882632492, 0.1771444480509246, 0.1044976975624465, 0.0747466038846997, 0.1044976975624465, -0.1159248155235217, 0.0840362919122998, 0.009640509993362]
functional_edDCA_batch2 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

norm_re_edDCA_batch3 = [0.1044976975624465, -0.0563696155664461, 0.0448188728117796, 0.0840362919122998, 0.0747466038846997, 0.0905578260461159, 0.1281186568679526, -0.0268919080867205, -0.0173117310179154, 0.1346401910017686, -0.0843072807027433, 0.0352386957429747, 0.1281186568679526, -0.0994912693917855, 0.025131763520361, 0.0939257181745546, 0.1158534787787101, 0.015650414525504, -0.1331048768720684, 0.0939257181745546, 0.0747466038846997, 0.2220664475640582, 0.2107106663477946, 0.025131763520361, 0.2889859699968454, 0.0939257181745546, 0.0939257181745546, 0.0659880530382574, 0.1260050657949795, 0.1239202319564615, 0.0577031746658874, 0.0010457570495767, 0.2107106663477946, -0.1331048768720684, 0.0219056880826045, 0.1359784783149381, 0.15605632200425, 0.0352386957429747, 0.0352386957429747, -0.081081205264987, 0.0980674006586821, 0.1044976975624465, 0.1414516645283228, -0.1936798031434509, 0.1158534787787101, -0.2253503322318661, -0.0709742730423732, 0.0284272222164206, 0.0448188728117796, -0.1376793336859963, -0.0965724587919859, -0.0626248891235466, 0.0352386957429747, 0.0284272222164206, 0.1158534787787101, 0.1198337784955828, 0.0659880530382574, 0.1346401910017686, 0.1281186568679526, 0.0460596320066608, 0.0793210606986275, 0.0939257181745546, 0.1902492606976479, 0.1809595726700477, 0.262269290789598, 0.1044976975624465, -0.1079282400082495, 0.0010457570495767, 0.0840362919122998, -0.0221766768730481, 0.0659880530382574, 0.1612511558734365, 0.0604153326067938, -0.0221766768730481, 0.1081902358363561, 0.0956378356886146, -0.0444233406585818, 0.1044976975624465, -0.0359082099162994, -0.1547227629048086, 0.0387614468268319, 0.136877207714395, -0.0563696155664461, -0.128389645658396]
functional_edDCA_batch3 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

norm_re_bmDCA_batch1 = [0.0939257181745546, 0.3282794163494062, 0.7715117067865883, -0.0674515219585161, -0.0221766768730481, 0.4884267614035454, 0.8493128859506193, 0.9928772705912644, -0.0129667768833583, -0.0994912693917855, 0.0939257181745546, -0.0236065434536627, 0.0460596320066608, 0.825346381977415, 0.0209265606403474, 0.1953745854146584, 0.0093532879348633, -0.0563696155664461, -0.0536318026448711, -0.054718975655355, 0.8007202836892512, 1.201761507601075, 1.0848362340125812, 0.0547082990740345, 0.0617896281267662, -0.0709742730423732, 0.0116177364374685, 0.1044976975624465, -0.0674515219585161, -0.0935969687303435, 0.1100704179939101, 0.3101121616065886, -0.1547227629048086, 0.0240488408270006, 1.038879979578757, -0.0118222034455152, -0.0299566687150269, 0.0423670937327512, 0.0659880530382574, 0.8321300049310648, 1.1879468230674588, 0.0985345745673327, -0.0533297283938141, 0.1349577723037267, 0.0894514463418647, 1.075041913310775, 0.0128472950775531, -0.090562554259844, 0.0468089387484024, 0.0041730681117937, 0.0219056880826045, -0.0100502338981669, -0.0679785488696036, 0.351607443194559, 0.0341708661718471, -0.0070924167416067, 0.6609071739004286, 0.4554451564755872, 0.5669312380779825, 0.5555459358819644, -0.0173823290027371, -0.0459219631008671, 0.4130094228787553, 0.0467476333171608, 0.3867949211605835, -0.1600071804791744, 0.020317767931149, -0.0698896469565958, 0.057300459786667, 0.0481501545175513, -0.0485097941194605, 1.0872724160934073, -0.0028059016159362, 0.0648571776224449, 0.1485800625180992, -0.0007170076053656, 1.0059919767397325, -0.0653585519305813, 0.0893632801004539, 0.5978043033487442, -0.0097118467381736, 0.6248334806694511, -0.0440086369593318, 0.983126769898806, -0.0148200838696248, 0.0781645803160341, 0.0840362919122998, 0.0126160000550044, -0.0638458750525968, -0.0135589004263504, 0.6718014831918174, -0.051614474748864, 0.4050146776550287, 0.0169695009209934, -0.0753876863066961, -0.0314663649006483, -0.0843072807027433, 0.0126160000550044, 0.1198337784955828, -0.0524900938363332, -0.1365484582701838, 0.6816370963109925, 1.004835756673354, 0.0129698868712914, -0.1848292481373278, -0.2947248943308306, 0.14497441561218, 0.0346593645298599, 1.0605340831133063, -0.0208023813530757, 0.8016422244206378, 1.089840088745334, -0.0129051261388755, 0.4475971977814776, 0.2904523249508541, -0.1106403979491559, -0.0197635100969439, 0.0231879794428964, -0.0146106061564208, 0.0044667260484265, 0.0329343978335067, 0.3176314097094736, 0.0460596320066608, 1.04893156606186, 0.0001198755417215, -0.0563696155664461, 0.9589940498877848, 0.0138225647404934, 0.8348073201754017, 0.0747466038846997, -0.1464378845324387, 0.8433312753936987, 0.0747466038846997, -0.0333858003113166, 1.1189517244206988, 1.061396380278632, 1.0525280636410903, 1.0896654332899556, 0.0284272222164206, -0.0329613321081889, 0.0179931455737097, -0.1051672117357713, -0.0444233406585818, 0.3452979860119617, -0.0764474592557579, 0.0939257181745546, 0.0537228749490147, -0.0063357094095207, 0.1765177276543966, 0.2181869258339453, 0.0128472950775531, 0.1158534787787101, 0.0916722522188934, 0.1129346681789105, 0.8891206168977861, 0.0498433532189019, 0.0770167627891596, 0.2876238905986432, 0.0219056880826045, 0.9718781091525012, -0.0314663649006483, 0.0304140626720963, 0.9146221388777904, -0.0329613321081889, -0.0863254292791118, 0.1461668957419951, -0.0524900938363332, -0.0108829207227375, 0.6029441348974919, 1.0801414889721157, 0.0659880530382574, 0.0770167627891596, 0.6532686383035372, -0.0143168554260626, -0.0122872506107933, -0.0650572088180365, 0.0715034579937039, 0.0531626076947043, 0.8782226608368816, 0.0325829460203916, 0.0209962912096256, 0.0423670937327512, 0.1044976975624465, 0.843377470834727, 0.7738567099870507, 0.0019772670510081, 0.8452177090331595, 0.8198911131031427, 1.0367002096468474, 0.9728544475617452, 0.2220664475640582, -0.0764474592557579, -0.0410738537819032, 0.0183000411766854, -0.0335483659972775, 0.0150387054363183, -0.1454890708300862, -0.0444233406585818, 0.0165285425638995, 0.0391579099803281, 1.0997064454132748, 0.2145527731287911, 0.1287279337078171, 0.1765177276543966, -0.1376793336859963, 0.9062647453033864, 1.0649759281289577, 0.0352386957429747, 0.0864494586884039, -0.0100933080338659, 0.203582268358018, 0.0600249300431434, 0.8762436582510605, 0.3649306899617098, 1.027054706940237, 0.9029276649839042, 1.0963452755154148, 0.8958696498066626, -0.12745570244209, -0.0160709718230344, -0.0744178544404887, 0.2107106663477946, -0.128389645658396, 1.0612765820747263, 0.0623824061323381, 0.2271258568346424, 0.1290334830081508, 0.0047330682960312, -0.1235246998032635, 0.4168634616746057, 0.030664238929047, 0.0864494586884039, 0.077343834254521, 1.0433357735949174]
functional_bmDCA_batch1 = [0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1]

norm_re_bmDCA_batch2 = [-0.0285413634468249, 0.0150387054363183, 0.15605632200425, 0.0517708367885755, -0.2071430555403872, 0.0939257181745546, -0.0234771055267673, -0.0139804493121439, 0.0329343978335067, -0.1856553036329587, -0.0477011752332412, -0.0485097941194605, 0.9913694888646508, -0.0630329663909444, -0.0598232118610288, -0.1440247177563346, -0.0131361886456058, 0.0808767382569404, 0.0392713738310101, 0.1170367504758827, 0.0344667398365557, 0.8587573069001487, -0.0366361586646489, 0.9929066672821784, -0.1075066911224558, 1.1446199776523598, 0.0939257181745546, -0.1159248155235217, -0.0608989972444862, 0.11163912611674, 0.0113337086947126, -0.1079282400082495, 0.0628284993828978, 0.0442022404973744, 0.1239202319564615, -0.1587030626216811, -0.03274865626094, 0.0409146380869604, 1.08325924299866, -0.0187175475145577, -0.196775523045192, -0.0435928078755293, 0.0360603219653373, 0.1450502547245149, -0.1926338405610645, 0.015650414525504, 1.0967371912083654, 0.9321575883032688, -0.0546747142675993, 0.0212231349569675, -0.2221377843088698, -0.0324646285181841, -0.0794423348476108, -0.0945952323478794, -0.0351767864590904, -0.0348097613175257, -0.0108618177890466, 0.0548828571981078, 0.0559598916616411, 0.0609635726311353, -0.1714556896397775, -0.1654290982183137, 0.0604153326067938, 0.277244267956694, 0.2695099884717184, -0.1065539444882771, -0.0851032976790625, 0.0211979135062736, 0.0138225647404934, -0.0268919080867205, 0.0609047387185888, -0.0342575700052083, 0.3308889331696253, 0.0303125451245696, -0.0122872506107933, 0.0550381870880884, 0.0141256962406259, -0.0343551602367278, 0.1818636043385183, 1.0872507623205536, 0.0239140120252135, -0.053956448790342, 0.1080204486463037, 0.0756506355531702, 0.0607577526897215, 0.0110615364680152, -0.0601533367786872, 0.2138702200031541, 0.0171904568689323, 0.0550381870880884, -0.0975515862342431, 0.0278227523069414, 0.0725095871720733, 0.1246120296251526, 0.1296586992113809, 0.0352386957429747, 0.121863383310852, -0.1388019242258451, -0.0351767864590904, -0.0006168226241277, 0.0309461763100469, -0.0314663649006483, 0.0506340888205884, 0.6551668578712286, 0.0577031746658874, -0.0380817630026947, 0.8574996553313139, 0.071182886907444, 0.1530872381351854, -0.0450138343501825, 0.0783951791907723, 0.1971631968982604, -0.0070924167416067, -0.0524900938363332, 0.0607577526897215, 0.0725095871720733, -0.0657794391731866, 0.012816435562124, 0.2001386869599027, -0.2113801805211192, -0.0017152712229015, -0.0999037404977573, 0.1722010218236054, -0.0290436595193343, 0.0476698001325066, 0.0352386957429747, 0.2241968303660871, 0.2442210519155554, -0.007970544780002, 0.922933737066544, 0.1260890520526833, 1.108904924095338, 0.0998772593758273, 0.0223319273653542, -0.2748975171033204, 0.0352386957429747, 0.015650414525504, 0.2151525113634458, 0.0840362919122998, 1.0301805117924083, 0.0577031746658874, 0.3904100201791015, 0.1346401910017686, -0.0122872506107933, 0.0649276133503432, 0.025131763520361, 0.2095825661114762, 0.1461668957419951, 0.1044976975624465, 0.0595059357338149, 0.07575141353993, 0.9216135992202033, 0.262269290789598, 1.040270800271167, 0.8453386590732738, 0.1158534787787101, -0.1210330526549729, -0.0001436377189976, 0.1722010218236054, 0.1328737976855347, 0.1249591032125932, -0.0314663649006483, 0.0905578260461159, -0.0221766768730481, 1.1200931536149292, 0.818805459510163, -0.0221766768730481, -0.1179885897944671, 0.0872623673500562, 0.009640509993362, -0.0268919080867205, 1.1298184567224452, 0.0450437194014185, 0.0537228749490147, 1.091384540126502, -0.0610848467801186, 0.009640509993362, 0.1299538036325758, -0.0765314455134617, 0.1023241444760512, 0.9132786365449376, -0.0348097613175257, 0.1324353626987439, 0.0577031746658874, -0.0196203686354889, -0.0444233406585818, 0.1612511558734365, 0.015650414525504, 0.0211096711062855, 0.0294298403409359, 0.030664238929047, 0.0747466038846997, 0.2107106663477946, 0.0577031746658874, 0.0505414543049141, 0.0255982263565141, 1.105715879595944, 0.003857449208562, -0.0592161294329656, -0.173664490743864, 0.0166753877340688, -0.0471597155767562, 0.0995275454392988, 0.0219056880826045, -0.090562554259844, 0.1485800625180992, 0.0537228749490147, -0.0474983520092617, 0.031309431138429, 0.2211624158955876, 0.2343316256533006, 0.1100704179939101, -0.0130151993591426, 0.2053335208290894, 0.0412486002751166, 0.5703136020211076, 0.0747466038846997, -0.1421211787016475, 1.0212025026378362, 0.1394744380842162, -0.0250231907395675, 0.1456357585608375, 0.3936588964387651, 0.0840362919122998, -0.081081205264987, 0.003857449208562, -0.0608313596696318, 0.0229735176537321, -0.1204037584508284, 0.1510318415971277, 0.0537228749490147, -0.0409326903234217, 0.2701291122365836, 0.1930102889701261, -0.0245524287000358, 0.0738478744852428, 0.1722010218236054, 0.4358480376514059, 0.4851001963710592, 0.015650414525504, 0.0852381249229642, 0.0577031746658874, 0.0226167470074764, -0.1092903194064516]
functional_bmDCA_batch2 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0]

norm_re_bmDCA_batch3 = [-0.0363938931227299, 0.0889012377674325, 0.0284272222164206, -0.1197718692116983, 0.0335072318589192, -0.0211088473019205, -0.1068297914094758, 0.0524187570915216, -0.0359082099162994, 0.0889012377674325, 0.0019772670510081, -0.0935969687303435, 0.1461668957419951, 0.0498433532189019, -0.085895200854199, -0.0453409058155438, 0.0991205520437413, 0.1239202319564615, -0.0762980365980426, -0.0237645970245036, -0.1557275725600388, -0.0329613321081889, -0.1682236970733216, 0.014710332292617, 0.1809595726700477, 0.1158534787787101, -0.1485514756054118, -0.0100824223077684, -0.090562554259844, 0.1239202319564615, -0.1079282400082495, 0.0048242216479225, -0.0572511113243906, -0.1159248155235217, 0.0659880530382574, -0.0536085872939678, 0.1281186568679526, -0.2000601137726423, 0.2380241639272103, -0.0070924167416067, 0.0905578260461159, 0.0997087789593548, 0.0747466038846997, 0.0073010306066775, 0.1044976975624465, 0.0423670937327512, 0.1263052283265032, 0.1535234887454185, -0.0349671654642323, 0.0840362919122998, -0.0221766768730481, 0.1100704179939101, 0.0927948427587423, 0.1414516645283228, 0.2260467472809307, 0.0540254079660333, 0.0010457570495767, 0.0659880530382574, -0.0321581625693393, 0.052939073120643, -0.0173117310179154, 0.0747466038846997, -0.0806147424288338, 0.0747466038846997, 0.0506970207379151, -0.039157086175963, 0.0617896281267662, -0.0915807169649145, 0.0233862109367063, 0.0703047588690486, 0.0317951143448593, -0.2967332182734394, 0.1158534787787101, 0.0448188728117796, -0.0182971551429351, -0.1185002193961414, 0.0352386957429747, -0.0983304083598588, 0.0309461763100469, -0.2234308968211976, 0.1281186568679526, 0.0329343978335067, -0.0197635100969439, 0.2343316256533006, 0.1500464174721082, -0.0205721289831631, -0.2141412087935975, 0.0509418018176756, 0.7565085692579149, 0.1182292306056979, -0.0197635100969439, 0.0485717034033448, -0.1009300867550391, -0.0268919080867205, 0.2107106663477946, 1.007664366836858, -0.0268919080867205, 0.15605632200425, -0.0843072807027433, -0.0122872506107933, 0.0577031746658874, -0.0656593035940462, 0.1530219075337505, 0.0070432796235409, 0.0772248183857458, -0.0563696155664461, -0.1133053855269547, -0.0163199286988286, -0.081081205264987, 0.0352386957429747, -0.1464378845324387, 0.0747466038846997, 0.0626201609098186, -0.1718722723793943, -0.0105751330967334, -0.0197635100969439, 0.0939257181745546, 0.036989948214046, -0.1547227629048086, 0.0537228749490147, -0.0620107282879736, 0.0991205520437413, -0.0070924167416067, 0.5262538528020976, 0.0197920970096315, 0.0747466038846997, 0.0939257181745546, 0.1380080831302074, -0.0099114987838055, -0.0870194386436498, 0.0084663056013399, 0.0352386957429747, 0.1827730012114973, 0.2685245643466987, -0.0430923188404633, -0.0885057056142345, 0.0604153326067938, 0.1100704179939101, 0.2181869258339453, 0.0144294285964539, -0.058273154621133, 0.15605632200425, 0.0659880530382574, -0.0563696155664461, 0.1158534787787101, 0.1008920506565272, -0.1181349440781284, 0.0271005219517913, -6.463131181045267e-05, 0.0867973201847781, -0.0638458750525968, -0.0563696155664461, 0.1081902358363561, -0.1331048768720684, -0.1060930932436263, 0.0273208425121696, 0.0897272932630636, -0.0303354894848358, 0.015650414525504, 0.1081902358363561, 0.0067216993935625, 0.0840362919122998, -0.0122872506107933, -0.1390254587999064, 0.0872623673500562, -0.0035286997643509, 0.0027265737927497, -0.1185002193961414, -0.0070924167416067, 0.0617896281267662, 0.0617896281267662, 0.0423670937327512, 0.0473105199600704, 0.0793210606986275, 0.0046703578702143, -0.0544421319967725, -0.1307653974853839, 0.1171687909177837, -0.0553446423578813, 0.0366380928967725, 0.1473687287526595, -0.0485097941194605, 0.0840362919122998, -0.0070924167416067, -0.0874668343581029, -0.081081205264987, 0.0905578260461159, -0.0314663649006483, 0.1666283013921418, 0.196770794831464, 0.015650414525504, 0.0187461344272451, 0.121863383310852, -0.0874668343581029, 0.0423670937327512, -0.1133053855269547, 0.14497441561218, 0.0284272222164206, -0.0353599698919579, 0.009640509993362, 0.0659880530382574, 0.0770167627891596, 0.0187461344272451, -0.009061175173037, -0.1376793336859963, -0.0502283122067486, -0.1110877936636089, -0.0402249157470906, -0.0202491933033744, 0.1612511558734365, -0.125976478882292, 0.0659880530382574, -0.0656593035940462, 0.0076884718329228, -0.0674515219585161, 0.1318111951418622, 0.15605632200425, 0.0659880530382574, 0.1281186568679526, 0.1092617991076909, 0.015650414525504, -0.0299566687150269, -0.0148200838696248, -0.0221766768730481, -0.0189506014352916, 0.1984454882585521, -0.0359082099162994, 0.0557001013931213, -0.0044274291638078, 0.003857449208562, 0.0284272222164206, 0.1941287824277609, -0.0777857465689274, 0.1044976975624465]
functional_bmDCA_batch3 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

## Enrichment Distributions and GMM Fits

Build the 4 by 3 panel figure. Each subplot shows the empirical normalized relative enrichment distribution, a two-component Gaussian mixture fit, the sample size, and the functional fraction for the corresponding model and divergence bin.


In [ ]:
# ============================================================================
# PLOTTING: GMM FITS AND HISTOGRAMS
# ============================================================================
# This cell builds a 4x3 grid of histograms (one row per model, one column per
# divergence batch). For each dataset we:
#  1) plot the normalized relative enrichment histogram (density normalized),
#  2) fit a 2-component GaussianMixture (sklearn) to the 1D data and plot its PDF,
#  3) annotate sample size `n` and fraction functional `p` on the panel,
#  4) add a vertical guide line at x=0.42 used in the manuscript for reference,
# The final figure is saved as 'images/experiment.pdf'.
# ============================================================================

models_cell7 = {
    "bmDCA": {
        1: (np.array(norm_re_bmDCA_batch1), np.array(functional_bmDCA_batch1)),
        2: (np.array(norm_re_bmDCA_batch2), np.array(functional_bmDCA_batch2)),
        3: (np.array(norm_re_bmDCA_batch3), np.array(functional_bmDCA_batch3)),
    },
    "eaDCA": {
        1: (np.array(norm_re_eaDCA_batch1), np.array(functional_eaDCA_batch1)),
        2: (np.array(norm_re_eaDCA_batch2), np.array(functional_eaDCA_batch2)),
        3: (np.array(norm_re_eaDCA_batch3), np.array(functional_eaDCA_batch3)),
    },
    "edDCA": {
        1: (np.array(norm_re_edDCA_batch1), np.array(functional_edDCA_batch1)),
        2: (np.array(norm_re_edDCA_batch2), np.array(functional_edDCA_batch2)),
        3: (np.array(norm_re_edDCA_batch3), np.array(functional_edDCA_batch3)),
    },
    "meDCA": {
        1: (np.array(norm_re_meDCA_batch1), np.array(functional_meDCA_batch1)),
        2: (np.array(norm_re_meDCA_batch2), np.array(functional_meDCA_batch2)),
        3: (np.array(norm_re_meDCA_batch3), np.array(functional_meDCA_batch3)),
    },
}

xmin_hist, xmax_hist = -0.5, 1.15
n_bins = 32
bin_edges = np.linspace(xmin_hist, xmax_hist, n_bins + 1)

xmin_plot, xmax_plot = -0.55, 1.2

batch_labels = {
    1: r"$[0.2\!-\!0.25]$ seq. divergence",
    2: r"$[0.4\!-\!0.45]$ seq. divergence",
    3: r"$[0.6\!-\!0.65]$ seq. divergence",
}

fig, axes = plt.subplots(4, 3, figsize=(22, 18), sharey=True)

# Keep explicit spacing knobs for manual tuning.
SUBPLOT_WSPACE = 0.02
SUBPLOT_HSPACE = 0.02

for row, (model, batches) in enumerate(models_cell7.items()):
    for col, (batch_id, (norm_re, func)) in enumerate(batches.items()):
        ax = axes[row, col]

        ax.hist(
            norm_re,
            bins=bin_edges,
            alpha=0.7,
            color=palette[0],
            edgecolor="white",
            label="Data",
            density=True,
        )

        X = norm_re.reshape(-1, 1)
        gmm = GaussianMixture(n_components=2, random_state=0)
        gmm.fit(X)

        x = np.linspace(xmin_plot, xmax_plot, 500).reshape(-1, 1)
        pdf = np.exp(gmm.score_samples(x))
        ax.plot(x, pdf, color="#1F4E79", lw=3)

        ax.set_xlim(xmin_plot, xmax_plot)
        ax.set_xticks([-0.5, 0.0, 0.5, 1.0])
        ax.tick_params(axis="x", labelsize=30)
        ax.tick_params(axis="y", labelsize=30)

        if row == 0:
            ax.set_title(batch_labels[batch_id], fontsize=36, fontweight="bold", pad=15)

        if col == 0:
            ax.text(
                -0.065,
                0.5,
                model,
                transform=ax.transAxes,
                fontsize=40,
                fontweight="bold",
                ha="right",
                va="center",
                rotation=90,
            )

        if row == len(models_cell7) - 1:
            ax.set_xlabel("normalized relative enrichment", fontsize=30, fontweight="bold")
        else:
            ax.set_xticklabels([])

        ax.text(
            0.02,
            0.95,
            f"n={len(norm_re)}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=30,
            color="black",
            fontweight="bold",
        )

        frac_func = func.mean()
        ax.text(
            0.95,
            0.5,
            f"p={frac_func:.3f}",
            transform=ax.transAxes,
            ha="right",
            va="center",
            fontsize=30,
            color="black",
            fontweight="bold",
        )

        ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
        ax.axvline(x=0.42, color="#1F4E79", linestyle="--", lw=2.5)

plt.tight_layout()
fig.subplots_adjust(wspace=SUBPLOT_WSPACE, hspace=SUBPLOT_HSPACE)

plt.savefig(output_path + "experiment.pdf")
plt.show()